# DATA 2402: Assignment 2 — Wisconsin Breast Cancer Predictor
**Course:** DATA 2402  
**Topic:** OOP Architecture, Missing-Value Imputation, Nearest-Neighbor Classification, and Custom Exceptions1 Handling

### Demonstrated Test Cases:
1. Load CSV (float conversion)
2. DataTable query & statistical methods
3. Missing historical values imputation
4. Missing new-case values imputation
5. Multiple missing values in same row
6. Majority-Class (Mode) Predictor
7. Nearest Neighbor Predictor (Euclidean distance on 9 features)
8. OutOfSampleError on single out-of-range feature
9. OutOfSampleError on multiple out-of-range features (catches first)
10. Final model predictions & accuracy evaluation

In [2]:
import csv
import math
from data_representation import DataTable
from exceptions1 import OutOfSampleError
from predictors import Predictor, MajorityClassPredictor, NearestNeighborPredictor
from main7 import calculate_accuracy

print("All custom modules imported successfully.")

All custom modules imported successfully.


## Tests 1 & 2: DataTable Ingestion and Query Methods
- **Test 1:** Validates numeric conversion to `float`.
- **Test 2:** Tests `get()`, `get_row()`, `get_column()`, `get_column_names()`, `get_column_average()`, `get_column_min()`, and `get_column_max()`.

In [3]:
# Create a small synthetic dataset to verify statistical calculations
synthetic_csv = "test_synthetic.csv"
with open(synthetic_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Feature1", "Feature2", "TumorType"])
    writer.writerow([1.0, 10.0, 0])
    writer.writerow([3.0, 20.0, 0])
    writer.writerow([5.0, 30.0, 1])

dt_test = DataTable(synthetic_csv)

# Test 1 Verification
assert isinstance(dt_test.get(0, "Feature1"), float), "Values must be float!"
print("Test 1 Passed: CSV loaded with float data conversion.")

# Test 2 Verification
print("\n--- Test 2: Method Verification ---")
print("get_column_names():", dt_test.get_column_names())
print("get(1, 'Feature2'): ", dt_test.get(1, "Feature2"))
print("get_row(0):        ", dt_test.get_row(0))
print("get_column('F1'):  ", dt_test.get_column("Feature1"))
print("Average Feature1:  ", dt_test.get_column_average("Feature1"))
print("Min Feature1:      ", dt_test.get_column_min("Feature1"))
print("Max Feature1:      ", dt_test.get_column_max("Feature1"))

assert dt_test.get_column_average("Feature1") == 3.0
assert dt_test.get_column_min("Feature1") == 1.0
assert dt_test.get_column_max("Feature1") == 5.0
print("Test 2 Passed: All DataTable query methods verified.")

Test 1 Passed: CSV loaded with float data conversion.

--- Test 2: Method Verification ---
get_column_names(): ['Feature1', 'Feature2', 'TumorType']
get(1, 'Feature2'):  20.0
get_row(0):         {'Feature1': 1.0, 'Feature2': 10.0, 'TumorType': 0.0}
get_column('F1'):   [1.0, 3.0, 5.0]
Average Feature1:   3.0
Min Feature1:       1.0
Max Feature1:       5.0
Test 2 Passed: All DataTable query methods verified.


## Tests 3, 4, & 5: Missing-Value Imputation Policy
- **Test 3:** Replace `?` with the mean of valid values in the same historical feature column.
- **Test 4:** Replace `?` in new cases using the historical feature mean.
- **Test 5:** Replace multiple missing values without altering row count or order.

In [4]:
# Test 3: Historical CSV missing value imputation
hist_missing_csv = "test_hist_missing.csv"
with open(hist_missing_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Feature1", "Feature2", "TumorType"])
    writer.writerow([2.0, 10.0, 0])
    writer.writerow(["?", 20.0, 0])  # Missing F1; valid average = (2.0 + 6.0) / 2 = 4.0
    writer.writerow([6.0, "?", 1])   # Missing F2; valid average = (10.0 + 20.0) / 2 = 15.0

dt_hist = DataTable(hist_missing_csv)
print("Test 3 - Historical Imputation:")
print(f"Row 1 Feature1: {dt_hist.get(1, 'Feature1')} (Expected: 4.0)")
print(f"Row 2 Feature2: {dt_hist.get(2, 'Feature2')} (Expected: 15.0)")
assert dt_hist.get(1, "Feature1") == 4.0
assert dt_hist.get(2, "Feature2") == 15.0
print("Test 3 Passed: Historical missing values correctly imputed using valid column means.")

# Test 4 & 5: New cases with multiple missing values
new_missing_csv = "test_new_missing.csv"
with open(new_missing_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Feature1", "Feature2"])
    writer.writerow(["?", "?"])    # Multiple missing values in one row
    writer.writerow([5.0, "?"])
    writer.writerow(["?", 12.0])

dt_new = DataTable(new_missing_csv, historical_averages=dt_hist.column_averages)
print("\nTest 4 & 5 - Multiple Missing Values in New Cases:")
print(f"Row count preserved: {len(dt_new.data)} (Expected: 3)")
print(f"Row 0 imputed: {dt_new.get_row(0)} (Expected F1=4.0, F2=15.0)")
assert len(dt_new.data) == 3
assert dt_new.get(0, "Feature1") == 4.0 and dt_new.get(0, "Feature2") == 15.0
assert dt_new.get(1, "Feature2") == 15.0 and dt_new.get(2, "Feature1") == 4.0
print("Test 4 & 5 Passed: Row count preserved and historical means applied.")

Test 3 - Historical Imputation:
Row 1 Feature1: 4.0 (Expected: 4.0)
Row 2 Feature2: 15.0 (Expected: 15.0)
Test 3 Passed: Historical missing values correctly imputed using valid column means.

Test 4 & 5 - Multiple Missing Values in New Cases:
Row count preserved: 3 (Expected: 3)
Row 0 imputed: {'Feature1': 4.0, 'Feature2': 15.0} (Expected F1=4.0, F2=15.0)
Test 4 & 5 Passed: Row count preserved and historical means applied.


## Test 6: Majority-Class (Mode) Predictor
Validates that `MajorityClassPredictor` identifies the most frequent outcome in historical data and predicts this class for incoming cases.

In [5]:
historical_data = DataTable("cancer_historical1.csv")
mode_model = MajorityClassPredictor(historical_data)

tumor_counts = historical_data.get_column("TumorType")
benign_count = tumor_counts.count(0.0)
malignant_count = tumor_counts.count(1.0)

print(f"Historical Benign (0):    {benign_count}")
print(f"Historical Malignant (1): {malignant_count}")
print(f"Identified Majority:      {mode_model.majority_class}")

pred = mode_model.predict(historical_data.get_row(0))
expected_majority = 0 if benign_count > malignant_count else 1
assert mode_model.majority_class == expected_majority
assert pred == expected_majority
print("Test 6 Passed: Mode predictor targets most common outcome.")

Historical Benign (0):    379
Historical Malignant (1): 220
Identified Majority:      0
Test 6 Passed: Mode predictor targets most common outcome.


## Test 7: Nearest Neighbor Predictor
Validates that `NearestNeighborPredictor` calculates Euclidean distance using only the 9 cell features (excluding `TumorType`) to locate the closest historical instance.

In [6]:
nn_model = NearestNeighborPredictor(historical_data)

p0 = historical_data.get_row(0)
self_dist = nn_model.distance(p0, p0)
print(f"Euclidean distance to self: {self_dist:.4f} (Expected: 0.0)")
assert self_dist == 0.0

pred_p0 = nn_model.predict(p0)
print(f"Prediction for Historical Patient 0: {pred_p0} (Actual: {p0['TumorType']})")
assert pred_p0 == p0['TumorType']
print("Test 7 Passed: 1-NN correctly computes Euclidean distance across 9 features.")

Euclidean distance to self: 0.0000 (Expected: 0.0)
Prediction for Historical Patient 0: 0 (Actual: 0.0)
Test 7 Passed: 1-NN correctly computes Euclidean distance across 9 features.


## Tests 8 & 9: Custom Exceptions1 (`OutOfSampleError`)
- **Test 8:** Raises `OutOfSampleError` when a feature falls outside `[min, max]` with a meaningful error message.
- **Test 9:** Raises `OutOfSampleError` immediately for the *first* out-of-range feature encountered in sequence.

In [7]:
# Test 8: Single out-of-range feature
f1_max = historical_data.get_column_max("Feature1")
patient_invalid = historical_data.get_row(0).copy()
patient_invalid["Feature1"] = f1_max + 10.0

try:
    mode_model.predict(patient_invalid)
    assert False, "Failed to raise OutOfSampleError!"
except OutOfSampleError as e:
    print(f"Test 8 Passed -> Caught Expected Exceptions1: {e}")

# Test 9: Multiple out-of-range features (Feature2 and Feature4)
patient_multi_invalid = historical_data.get_row(0).copy()
patient_multi_invalid["Feature2"] = historical_data.get_column_max("Feature2") + 5.0
patient_multi_invalid["Feature4"] = historical_data.get_column_max("Feature4") + 5.0

try:
    nn_model.predict(patient_multi_invalid)
    assert False, "Failed to raise OutOfSampleError for multiple invalid features!"
except OutOfSampleError as e:
    print(f"Test 9 Passed -> First Invalid Feature Caught: {e}")
    assert "Feature2" in str(e), "Expected Feature2 to trigger first!"

Test 8 Passed -> Caught Expected Exceptions1: Feature1 with value 20.0 is outside historical range [1.0, 10.0]
Test 9 Passed -> First Invalid Feature Caught: Feature2 with value 15.0 is outside historical range [1.0, 10.0]


## Test 10: Model Evaluation & Accuracy Calculation
Load the 100 new cases and the 100 corresponding ground-truth diagnosis labels. Generate predictions using both models, track valid predictions, and compute overall accuracy.

In [8]:
new_cases = DataTable("cancer_new_cases1.csv", historical_averages=historical_data.column_averages)
labels_table = DataTable("cancer_new_cases_labels1.csv")
actual_labels = labels_table.get_column("TumorType")

mode_predictions = []
nn_predictions = []
valid_actual = []
out_of_sample_count = 0

for i in range(len(new_cases.data)):
    patient = new_cases.get_row(i)
    try:
        p_mode = mode_model.predict(patient)
        p_nn = nn_model.predict(patient)
        
        mode_predictions.append(p_mode)
        nn_predictions.append(p_nn)
        valid_actual.append(actual_labels[i])
    except OutOfSampleError as err:
        out_of_sample_count += 1
        print(f"Case {i} Out-Of-Sample: {err}")

mode_acc = calculate_accuracy(mode_predictions, valid_actual)
nn_acc = calculate_accuracy(nn_predictions, valid_actual)

print("\n================ FINAL EVALUATION ================")
print(f"Total Test Cases Evaluated:       {len(new_cases.data)}")
print(f"Valid In-Sample Cases:            {len(valid_actual)}")
print(f"Out-Of-Sample Exceptions1 Handled: {out_of_sample_count}")
print(f"Majority Class (Mode) Accuracy:   {mode_acc:.2f}%")
print(f"Nearest Neighbor (1-NN) Accuracy: {nn_acc:.2f}%")
print("==================================================")

assert len(mode_predictions) == len(valid_actual)
assert len(nn_predictions) == len(valid_actual)
print("Test 10 Passed: Both models successfully evaluated against ground truth labels.")


================ FINAL EVALUATION ================
Total Test Cases Evaluated:       100
Valid In-Sample Cases:            100
Out-Of-Sample Exceptions1 Handled: 0
Majority Class (Mode) Accuracy:   79.00%
Nearest Neighbor (1-NN) Accuracy: 98.00%
Test 10 Passed: Both models successfully evaluated against ground truth labels.
